In [1]:
import sys
sys.path.append('../src')
import time

from market import Market
from naive import naive                                                  # Naive method (baseline)
from prod import prod                                                    # Prod method (baseline)
from tu_matching import tu_matching                                      # TU matching method (baseline)
from iter_lp import iter_lp                                              # IterLP method (baseline)
from alternate_fw import sw_maximize, nsw_maximize, alpha_sw_maximize    # SW/NSW/alpha-SW with LP
from alternate_fw_sinkhorn import sw_sinkhorn, nsw_sinkhorn              # SW/NSW with Sinkhorn algorithm

In [2]:
# Make a market with 30 left users and 20 right users
m = Market(num_left=30, num_right=20, v_left_type="inv", v_right_type="inv")    # Set examination functions "inv" or "log"
m.generate_preferences(pref_seed=0, lambda_value=0.5)    # Generate user preferences with lambda in [0.0, 1.0]

In [3]:
# Naive method
print("===== Naive method =====")

start = time.time()
stochastic_policy_for_left = naive(m.pref_left_to_right)
stochastic_policy_for_right = naive(m.pref_right_to_left)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)
print("Execution time:", exec_time)

===== Naive method =====
Expected number of matches: 6.097606001793401
Number of envies for left users: 383
Number of envies for right users: 172
Gini index for left users: 0.4530150639523828
Gini index for right users: 0.4723129992205657
Execution time: 0.0005660057067871094


In [4]:
# Prod method
print("===== Prod method =====")

start = time.time()
stochastic_policy_for_left = prod(m.pref_left_to_right, m.pref_right_to_left)
stochastic_policy_for_right = prod(m.pref_right_to_left, m.pref_left_to_right)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)

===== Prod method =====
Expected number of matches: 13.214715194901732
Number of envies for left users: 211
Number of envies for right users: 81
Gini index for left users: 0.4620551524029972
Gini index for right users: 0.4270392017833286
Execution time: 0.0007479190826416016


In [5]:
# TU matching
print("===== TU matching method =====")
start = time.time()
stochastic_policy_for_left, stochastic_policy_for_right = tu_matching(m.pref_left_to_right, m.pref_right_to_left, output=False)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)

===== TU matching method =====
Expected number of matches: 15.447528483780438
Number of envies for left users: 63
Number of envies for right users: 10
Gini index for left users: 0.34278589668452353
Gini index for right users: 0.21069573288772167
Execution time: 0.0021109580993652344


In [6]:
# IterLP
print("===== IterLP method =====")

start = time.time()
stochastic_policy_for_left, stochastic_policy_for_right = iter_lp(m.pref_left_to_right, m.pref_right_to_left)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)

===== IterLP method =====
Expected number of matches: 15.921955103345166
Number of envies for left users: 103
Number of envies for right users: 0
Gini index for left users: 0.34176502652527063
Gini index for right users: 0.1446208630075773
Execution time: 0.22331690788269043


In [7]:
# SW maximization
print("===== SW maximization with LP =====")

start = time.time()
stochastic_policy_for_left, stochastic_policy_for_right = sw_maximize(m.pref_left_to_right, m.pref_right_to_left, v_left=m.v_left, v_right=m.v_right, output=True)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)


===== SW maximization with LP =====
Step:001  SW:3.98908  UPDATE:3.98908  TIME:0.23258
Step:002  SW:4.53201  UPDATE:0.54293  TIME:0.47123
Step:003  SW:5.14751  UPDATE:0.61550  TIME:0.71868
Step:004  SW:5.80891  UPDATE:0.66140  TIME:0.95772
Step:005  SW:6.49300  UPDATE:0.68408  TIME:1.20179
Step:006  SW:7.18191  UPDATE:0.68892  TIME:1.49751
Step:007  SW:7.86176  UPDATE:0.67985  TIME:1.75186
Step:008  SW:8.52231  UPDATE:0.66055  TIME:2.04758
Step:009  SW:9.15645  UPDATE:0.63414  TIME:2.32012
Step:010  SW:9.75950  UPDATE:0.60305  TIME:2.59525
Step:011  SW:10.32854  UPDATE:0.56905  TIME:2.84951
Step:012  SW:10.86207  UPDATE:0.53353  TIME:3.11993
Step:013  SW:11.35963  UPDATE:0.49756  TIME:3.37903
Step:014  SW:11.82154  UPDATE:0.46191  TIME:3.64587
Step:015  SW:12.24870  UPDATE:0.42716  TIME:3.92976
Step:016  SW:12.64245  UPDATE:0.39375  TIME:4.18388
Step:017  SW:13.00437  UPDATE:0.36192  TIME:4.43595
Step:018  SW:13.33622  UPDATE:0.33185  TIME:4.69945
Step:019  SW:13.63987  UPDATE:0.30364 

In [8]:
# NSW maximization
print("===== NSW maximization with LP =====")
start = time.time()
stochastic_policy_for_left, stochastic_policy_for_right = nsw_maximize(m.pref_left_to_right, m.pref_right_to_left, v_left=m.v_left, v_right=m.v_right, output=True)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)

===== NSW maximization with LP =====
Step:001  SW:3.89666  UPDATE:3.89666  TIME:0.24730
Step:002  SW:4.38332  UPDATE:0.48666  TIME:0.48439
Step:003  SW:4.96279  UPDATE:0.57947  TIME:0.72775
Step:004  SW:5.59111  UPDATE:0.62832  TIME:0.98564
Step:005  SW:6.26409  UPDATE:0.67297  TIME:1.24857
Step:006  SW:6.91372  UPDATE:0.64963  TIME:1.56186
Step:007  SW:7.57720  UPDATE:0.66348  TIME:1.80945
Step:008  SW:8.20338  UPDATE:0.62618  TIME:2.06204
Step:009  SW:8.83494  UPDATE:0.63156  TIME:2.32015
Step:010  SW:9.40248  UPDATE:0.56754  TIME:2.57012
Step:011  SW:9.96476  UPDATE:0.56229  TIME:2.82474
Step:012  SW:10.46680  UPDATE:0.50204  TIME:3.07978
Step:013  SW:10.96721  UPDATE:0.50041  TIME:3.33195
Step:014  SW:11.41177  UPDATE:0.44456  TIME:3.61242
Step:015  SW:11.81967  UPDATE:0.40791  TIME:3.86350
Step:016  SW:12.20399  UPDATE:0.38431  TIME:4.11992
Step:017  SW:12.55896  UPDATE:0.35497  TIME:4.38620
Step:018  SW:12.89252  UPDATE:0.33356  TIME:4.64008
Step:019  SW:13.18408  UPDATE:0.29156 

In [9]:
for alpha in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    if alpha == 0.0:
        print("===== alpha :", alpha, "(NSW)", "=====")
    elif alpha == 1.0:
        print("===== alpha :", alpha, "(SW)", "=====")
    else:
        print("===== alpha :", alpha, "=====")

    start = time.time()
    stochastic_policy_for_left, stochastic_policy_for_right = alpha_sw_maximize(
        pref_left_to_right=m.pref_left_to_right,
        pref_right_to_left=m.pref_right_to_left,
        v_left=m.v_left,
        v_right=m.v_right,
        output=False,
        alpha=alpha
    )
    exec_time = time.time() - start

    res = m.get_match_prob(stochastic_policy_for_left, stochastic_policy_for_right)
    print("Expected number of matches:", res.sum())

    envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
    print("Number of envies for left users:", len(envy["left"]))
    print("Number of envies for right users:", len(envy["right"]))

    gini_left, gini_right = m.compute_gini(res)
    print("Gini index for left users:", gini_left)
    print("Gini index for right users:", gini_right)
    print("Execution time:", exec_time)
    print("")

===== alpha : 0.0 (NSW) =====
Expected number of matches: 15.762271452010426
Number of envies for left users: 0
Number of envies for right users: 0
Gini index for left users: 0.1495982300145933
Gini index for right users: 0.15393080490196273
Execution time: 11.040856838226318

===== alpha : 0.2 =====
Expected number of matches: 15.97161350642986
Number of envies for left users: 1
Number of envies for right users: 0
Gini index for left users: 0.15260729179410026
Gini index for right users: 0.16393887725496076
Execution time: 11.977122783660889

===== alpha : 0.4 =====
Expected number of matches: 16.26601506353349
Number of envies for left users: 8
Number of envies for right users: 1
Gini index for left users: 0.17319382188212368
Gini index for right users: 0.17186119790077906
Execution time: 12.557942867279053

===== alpha : 0.6 =====
Expected number of matches: 16.58421149744246
Number of envies for left users: 15
Number of envies for right users: 2
Gini index for left users: 0.2443328

In [10]:
# SW maximization with Sinkhorn algorithm
print("===== SW maximization with Sinkhorn =====")
start = time.time()
stochastic_policy_for_left, stochastic_policy_for_right = sw_sinkhorn(
    pref_left_to_right=m.pref_left_to_right,
    pref_right_to_left=m.pref_right_to_left,
    v_left=m.v_left,
    v_right=m.v_right,
    device="cpu",
    sinkhorn_lambda=200.0,
    output=True
)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)

===== SW maximization with Sinkhorn =====
Step:001  SW:3.84040  UPDATE:3.84040  TIME:0.01859
Step:002  SW:4.21897  UPDATE:0.37857  TIME:0.03546
Step:003  SW:4.67574  UPDATE:0.45677  TIME:0.05099
Step:004  SW:5.19198  UPDATE:0.51624  TIME:0.06650
Step:005  SW:5.74747  UPDATE:0.55550  TIME:0.08228
Step:006  SW:6.32446  UPDATE:0.57699  TIME:0.09790
Step:007  SW:6.90818  UPDATE:0.58372  TIME:0.11550
Step:008  SW:7.48712  UPDATE:0.57894  TIME:0.13244
Step:009  SW:8.05255  UPDATE:0.56543  TIME:0.14847
Step:010  SW:8.59809  UPDATE:0.54554  TIME:0.16482
Step:011  SW:9.11931  UPDATE:0.52122  TIME:0.18137
Step:012  SW:9.61331  UPDATE:0.49400  TIME:0.19810
Step:013  SW:10.07841  UPDATE:0.46510  TIME:0.21409
Step:014  SW:10.51384  UPDATE:0.43544  TIME:0.23025
Step:015  SW:10.91958  UPDATE:0.40574  TIME:0.24638
Step:016  SW:11.29613  UPDATE:0.37654  TIME:0.26185
Step:017  SW:11.64437  UPDATE:0.34824  TIME:0.27761
Step:018  SW:11.96549  UPDATE:0.32112  TIME:0.29293
Step:019  SW:12.26085  UPDATE:0.29

In [11]:
# NSW maximization with Sinkhorn algorithm
print("===== NSW maximization with Sinkhorn =====")
start = time.time()
stochastic_policy_for_left, stochastic_policy_for_right = nsw_sinkhorn(
    pref_left_to_right=m.pref_left_to_right,
    pref_right_to_left=m.pref_right_to_left,
    v_left=m.v_left,
    v_right=m.v_right,
    device="cpu",
    sinkhorn_lambda=200.0,
    output=True
)
exec_time = time.time() - start

res = m.get_match_prob(
    stochastic_policy_for_left=stochastic_policy_for_left,
    stochastic_policy_for_right=stochastic_policy_for_right
)
print("Expected number of matches:", res.sum())

envy = m.check_envy(stochastic_policy_for_left=stochastic_policy_for_left, stochastic_policy_for_right=stochastic_policy_for_right, match_prob=res)
print("Number of envies for left users:", len(envy["left"]))
print("Number of envies for right users:", len(envy["right"]))

gini_left, gini_right = m.compute_gini(res)
print("Gini index for left users:", gini_left)
print("Gini index for right users:", gini_right)

print("Execution time:", exec_time)


===== NSW maximization with Sinkhorn =====
Step:001  SW:3.85184  UPDATE:3.85184  TIME:0.01821
Step:002  SW:4.29177  UPDATE:0.43993  TIME:0.03541
Step:003  SW:4.81462  UPDATE:0.52286  TIME:0.05099
Step:004  SW:5.38684  UPDATE:0.57221  TIME:0.06649
Step:005  SW:5.98116  UPDATE:0.59432  TIME:0.08305
Step:006  SW:6.58507  UPDATE:0.60391  TIME:0.09907
Step:007  SW:7.18041  UPDATE:0.59534  TIME:0.11562
Step:008  SW:7.76656  UPDATE:0.58615  TIME:0.13172
Step:009  SW:8.32492  UPDATE:0.55836  TIME:0.14811
Step:010  SW:8.86276  UPDATE:0.53784  TIME:0.16452
Step:011  SW:9.36529  UPDATE:0.50254  TIME:0.18046
Step:012  SW:9.84253  UPDATE:0.47724  TIME:0.19577
Step:013  SW:10.28225  UPDATE:0.43971  TIME:0.21172
Step:014  SW:10.69612  UPDATE:0.41387  TIME:0.22894
Step:015  SW:11.07374  UPDATE:0.37762  TIME:0.24488
Step:016  SW:11.42648  UPDATE:0.35274  TIME:0.26036
Step:017  SW:11.74661  UPDATE:0.32013  TIME:0.27634
Step:018  SW:12.04338  UPDATE:0.29678  TIME:0.29177
Step:019  SW:12.31209  UPDATE:0.2